# Buzzards Bay basins: map and daily precipitation time series

Daily precipitation over each Buzzards Bay v2 basin polygon (Joe Costa's BBNEP study-area coverage), computed from the 1 km NOAA MRMS radar/gauge QPE. A report date covers **7am to 7am US Eastern** local time (the CoCoRaHS convention), 2014-11-02 to present.

- **Polygons** (geometry): `basins/bbnep_subbasins_2026_v2.geojson` on GitHub. 66 polygons: for each embayment a `LAND` (drainage) and a `WATER` polygon, plus `Coastal Watershed` (land) and `Buzzard Bay Nonembayment` (open-bay water).
- **Time series** (no geometry): `basins_v2_precip_ts.parquet` on Cloudflare R2, written daily by `basin_precip/`. Join to the polygons on `basin_id = BBP_SYS_ID + "_" + TYPE`.

**Data gaps:** the MRMS store has many missing hours in Nov 2014 to 2015. `n_hours` counts the hours with data over each basin and `n_hours_expected` is the window length (24, or 23/25 on DST days). A day with `n_hours < n_hours_expected` is a lower bound, and NaN if `n_hours` is 0. See the completeness section at the end.

In [ ]:
%pip install -q hvplot geoviews geopandas pyarrow

In [ ]:
import geopandas as gpd
import holoviews as hv
import hvplot.pandas  # noqa: F401  (registers .hvplot)
import numpy as np
import pandas as pd
import panel as pn

hv.extension("bokeh")
pn.extension()

## Load

In [ ]:
POLYGONS = "https://raw.githubusercontent.com/rsignell/buzzards-bay-precip/main/basins/bbnep_subbasins_2026_v2.geojson"
SERIES = "https://r2-pub.openscicomp.io/buzzards-bay-precip/basins_v2_precip_ts.parquet"

basins = gpd.read_file(POLYGONS).rename(
    columns={"TMDL_BASIN": "basin", "BBP_SYS_ID": "sys_id", "TYPE": "type"})
basins["basin_id"] = basins["sys_id"] + "_" + basins["type"]

# Public bucket, no credentials. Cloudflare rejects Python's default User-Agent (HTTP 403),
# so pandas passes our own header through storage_options.
ts = pd.read_parquet(SERIES, storage_options={"User-Agent": "buzzards-bay-precip-notebook"})
ts = ts.merge(basins[["basin_id", "basin"]], on="basin_id", how="left")

print(f"{len(basins)} polygons; {ts.basin_id.nunique()} basins in the time series, {len(ts):,} rows")
print(f"{ts.report_date.nunique():,} report dates, {ts.report_date.min():%Y-%m-%d} to {ts.report_date.max():%Y-%m-%d}")
latest = ts[ts.report_date == ts.report_date.max()]
print(f"\nwettest basins on the latest report date ({ts.report_date.max():%Y-%m-%d}):")
latest.nlargest(5, "precip_mm")[["basin", "type", "precip_mm", "volume_m3", "volume_flux_m3s"]].round(2)

## Map: the basins

Land (drainage) polygons and water polygons. Hover for the name, system id and area; pan and zoom freely.

In [ ]:
basin_map = basins.hvplot(
    geo=True, tiles="CartoLight", c="type",
    cmap={"LAND": "#16a34a", "WATER": "#2563eb"},
    hover_cols=["basin", "sys_id", "type", "ACRES"],
    alpha=0.45, line_color="#374151", line_width=0.6,
    width=850, height=620, title="Buzzards Bay v2 basins (land and water polygons)")
basin_map

## Map: precipitation on a chosen day

Pick a report date and a variable. The color scale is fixed per variable (0 to its 99th percentile) so days are comparable. `volume_m3` and `volume_flux_m3s` scale with polygon area, so the big watersheds and the open bay dominate them; use `precip_mm` to compare depth.

In [ ]:
VARS = ["precip_mm", "precip_in", "volume_m3", "volume_flux_m3s"]
clim = {v: (0.0, float(np.nanpercentile(ts[v], 99))) for v in VARS}
dates = pd.DatetimeIndex(np.sort(ts.report_date.unique()))

date_w = pn.widgets.DateSlider(name="report date", start=dates.min().to_pydatetime(),
                               end=dates.max().to_pydatetime(), value=dates.max().to_pydatetime())
var_w = pn.widgets.Select(name="variable", options=VARS, value="precip_mm")


def precip_map(date, var):
    d = ts[ts.report_date == pd.Timestamp(date)][["basin_id", var, "n_hours", "n_hours_expected"]]
    g = basins.merge(d, on="basin_id", how="left")
    return g.hvplot(
        geo=True, tiles="CartoLight", c=var, cmap="viridis", clim=clim[var], colorbar=True,
        hover_cols=["basin", "type", var, "n_hours", "n_hours_expected"],
        alpha=0.75, line_color="#374151", line_width=0.4, width=850, height=620,
        title=f"{var}, 7am-7am ET ending {pd.Timestamp(date):%Y-%m-%d}")


pn.Column(pn.Row(date_w, var_w), pn.bind(precip_map, date_w, var_w))

## Time series: every basin

All 66 series, daily `precip_mm`, one line per basin: land (drainage) polygons in green, water polygons in blue. Hover a line for its basin name; use the toolbar to zoom. The two panels share a time axis.

In [ ]:
opts = dict(x="report_date", y="precip_mm", by="basin_id", legend=False, hover_cols=["basin"],
            alpha=0.6, line_width=1, width=1000, height=330, xlabel="", ylabel="precip (mm/day)")
land = ts[ts["type"] == "LAND"].hvplot.line(
    color="#16a34a", title="Daily precipitation: 33 land (drainage) polygons", **opts)
water = ts[ts["type"] == "WATER"].hvplot.line(
    color="#2563eb", title="Daily precipitation: 33 water polygons", **opts)
(land + water).cols(1)

## Time series: land vs. water totals

Total daily precipitation volume summed over all land (drainage) polygons versus all water polygons, in millions of m³ per day. Days with no data show as gaps.

In [ ]:
tot = (ts.groupby(["report_date", "type"]).volume_m3.sum(min_count=1)
         .unstack("type").div(1e6).rename_axis(columns=None).reset_index())
tot.hvplot.line(x="report_date", y=["LAND", "WATER"], color=["#16a34a", "#2563eb"],
                width=1000, height=320, ylabel="million m³ per day", xlabel="report date",
                title="Daily precipitation volume: land vs. water polygons", legend="top_left")

## Explore one basin

Pick any basin: daily depth, volume flux, and the fraction of the day's hours that had data.

In [ ]:
labels = (basins.assign(label=basins["basin"] + " (" + basins["type"] + ")")
          .set_index("basin_id")["label"].sort_values())
picker = pn.widgets.Select(name="basin", options={lab: bid for bid, lab in labels.items()},
                           value="BBNEMB_WATER")


def basin_panels(basin_id):
    d = ts[ts.basin_id == basin_id].assign(frac_hours=lambda x: x.n_hours / x.n_hours_expected)
    opts = dict(width=1000, height=210, xlabel="")
    p1 = d.hvplot.line(x="report_date", y="precip_mm", color="#2563eb", ylabel="mm/day",
                       title=labels[basin_id], **opts)
    p2 = d.hvplot.line(x="report_date", y="volume_flux_m3s", color="#0f766e", ylabel="m³/s", **opts)
    p3 = d.hvplot.area(x="report_date", y="frac_hours", color="#9ca3af", ylabel="hours with data",
                       ylim=(0, 1.05), **opts)
    return (p1 + p2 + p3).cols(1)


pn.Column(picker, pn.bind(basin_panels, picker))

## Data completeness

Share of basin-days that have every expected hour of MRMS data, per day and per year. The early record (Nov 2014 to 2015) is patchy; from 2016 it is nearly complete. For analysis, filter on `n_hours == n_hours_expected` or start in 2016.

In [ ]:
ts["complete"] = ts.n_hours == ts.n_hours_expected
by_day = ts.groupby("report_date").complete.mean().mul(100)
by_year = ts.groupby(ts.report_date.dt.year).complete.mean().mul(100).round(1)

(by_day.hvplot.line(width=1000, height=240, ylabel="% of basins complete", xlabel="report date",
                    title="Basins with every hour present, by day", color="#9333ea", ylim=(0, 105))
 + by_year.hvplot.bar(width=1000, height=240, ylabel="% of basin-days complete", xlabel="year",
                      title="by year", color="#9333ea", ylim=(0, 105))).cols(1)